# Phase 2: Legal NER: Strided GQA decoder

This is the **Legal NER** part of the full `strided` notebook (`notebooks/CS781_strided_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Fine Tuning

In [ ]:
need = {
    "setup (tokenizer, data, token)": ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "train_dataset", "val_dataset"],
    "strict scorer cell":             ["strict_scores", "print_scores"],
    "Phase-2 cell 1 (toolkit)":       ["train_ner", "NERModel", "write_test_predictions", "train_docs", "val_docs", "NL", "P2_DIR", "predict_docs"],
    "attached model cell":            ["CustomDecoderLM", "alternating_patterns"],
}
for step, names in need.items():
    miss = [n for n in names if n not in globals()]
    print(f"{'✅' if not miss else '❌'} {step}" + (f"   missing: {miss}" if miss else ""))

✅ setup (tokenizer, data, token)
✅ strict scorer cell
✅ Phase-2 cell 1 (toolkit)
✅ attached model cell


In [6]:
import collections

def strict_scores(gold, pred, labels):
    """
    gold, pred: {doc_id: [[start, end, label, (text)], ...]}
    Exact match on (doc_id, start, end, label). Macro over `labels`.
    Returns dict with macro_p / macro_r / macro_f1, per-label dict, and micro_f1.
    """
    def to_set(d):
        out = set()
        for did, ents in d.items():
            for e in ents:
                out.add((str(did), int(e[0]), int(e[1]), e[2]))
        return out

    G, P = to_set(gold), to_set(pred)
    tp, gp, pp = collections.Counter(), collections.Counter(), collections.Counter()
    for x in G: gp[x[3]] += 1
    for x in P: pp[x[3]] += 1
    for x in G & P: tp[x[3]] += 1

    per = {}
    for lab in labels:
        p = tp[lab] / pp[lab] if pp[lab] else 0.0
        r = tp[lab] / gp[lab] if gp[lab] else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        per[lab] = {"p": p, "r": r, "f1": f, "support": gp[lab], "pred": pp[lab], "tp": tp[lab]}

    n = len(labels)
    macro_p = sum(v["p"] for v in per.values()) / n
    macro_r = sum(v["r"] for v in per.values()) / n
    macro_f1 = sum(v["f1"] for v in per.values()) / n
    TP, PP, GG = sum(tp.values()), sum(pp.values()), sum(gp.values())
    mp, mr = (TP / PP if PP else 0.0), (TP / GG if GG else 0.0)
    micro_f1 = 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0
    return {"macro_p": macro_p, "macro_r": macro_r, "macro_f1": macro_f1,
            "micro_f1": micro_f1, "per_label": per}

def print_scores(res):
    print(f"MACRO  P={res['macro_p']:.4f}  R={res['macro_r']:.4f}  F1={res['macro_f1']:.4f}  | micro F1={res['micro_f1']:.4f}")
    print(f"{'label':10s} {'P':>7s} {'R':>7s} {'F1':>7s} {'gold':>6s} {'pred':>6s} {'tp':>6s}")
    for lab, v in sorted(res["per_label"].items(), key=lambda kv: -kv[1]["support"]):
        print(f"{lab:10s} {v['p']:7.3f} {v['r']:7.3f} {v['f1']:7.3f} {v['support']:6d} {v['pred']:6d} {v['tp']:6d}")

# ---------- self-test on toy data ----------
_L = ["JUDGE", "COURT", "DATE"]
_gold = {"d1": [[0, 5, "JUDGE", "Smith"], [10, 20, "COURT", "High Court"]],
         "d2": [[3, 9, "DATE", "1 May"]]}

# perfect predictions (with and without optional text field)
_r = strict_scores(_gold, {"d1": [[0, 5, "JUDGE"], [10, 20, "COURT", "x"]], "d2": [[3, 9, "DATE"]]}, _L)
assert abs(_r["macro_f1"] - 1.0) < 1e-9, _r

# boundary off by one and wrong label must both count as wrong
_r = strict_scores(_gold, {"d1": [[0, 6, "JUDGE"], [10, 20, "JUDGE"]], "d2": [[3, 9, "DATE"]]}, _L)
assert _r["per_label"]["JUDGE"]["tp"] == 0 and _r["per_label"]["COURT"]["tp"] == 0
assert _r["per_label"]["DATE"]["f1"] == 1.0
assert abs(_r["macro_f1"] - 1/3) < 1e-9, _r["macro_f1"]

# empty predictions -> zeros; duplicate predictions must not inflate counts
assert strict_scores(_gold, {}, _L)["macro_f1"] == 0.0
_r = strict_scores(_gold, {"d2": [[3, 9, "DATE"], [3, 9, "DATE"]]}, _L)
assert _r["per_label"]["DATE"]["pred"] == 1 and _r["per_label"]["DATE"]["p"] == 1.0
print("✅ strict scorer self-test passed")

✅ strict scorer self-test passed


In [7]:
import os, json, math, time, random, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

for name in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "strict_scores", "print_scores"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cell that defines it."

# ============================================================
# 0. Config
# ============================================================
WINDOW, STRIDE = 1024, 768            # tokens per window / step (256 overlap)
VAL_FRACTION, SEED = 0.15, 42
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0

P2_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive not mounted — saving Phase-2 files locally (lost if the runtime disconnects).")
    P2_DIR = "CS781_Capstone_Project_Phase_2"
os.makedirs(P2_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if use_bf16 else torch.float16
PAD_ID = tokenizer.token_to_id("[PAD]")
PAD_ID = 1 if PAD_ID is None else PAD_ID
def amp(): return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)

# ============================================================
# 1. Load Phase-2 files
# ============================================================
BASE = "Phase_2_LNER"
def _path(fn): return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"{BASE}/{fn}", repo_type="dataset", token=HF_TOKEN)
def _json(fn):
    with open(_path(fn), "r", encoding="utf-8") as f: return json.load(f)

text1, gold1 = _json("Train/fold1-text.json"), _json("Train/fold1-gold.json")
text2, gold2 = _json("Train/fold2-text.json"), _json("Train/fold2-gold.json")
text3 = _json("Test/fold3-text.json")
with open(_path("Train/labels.txt"), "r", encoding="utf-8") as f:
    LABELS = [l.split()[0] for l in f if l.strip()]
print(f"Labels ({len(LABELS)}): {LABELS}")
assert len(LABELS) == 12, "labels.txt did not parse into 12 labels — paste me the first lines of the file."

def as_text(v): return v.get("text") if isinstance(v, dict) else v

def parse_entity(e):
    if isinstance(e, (list, tuple)):
        return int(e[0]), int(e[1]), e[2], (e[3] if len(e) > 3 else None)
    if isinstance(e, dict):
        g = lambda *ks: next((e[k] for k in ks if k in e), None)
        return (int(g("start", "start_char", "begin")), int(g("end", "end_char")),
                g("label", "type", "entity"), g("text", "entity_text", "span"))
    raise ValueError(f"Unknown annotation format: {e!r}")

def doc_entities(gold, doc_id):
    v = gold.get(doc_id, [])
    if isinstance(v, dict):
        v = next((x for x in v.values() if isinstance(x, list)), [])
    return [parse_entity(e) for e in v]

# ---- offset convention: is the gold `end` exclusive (python slice) or inclusive? ----
excl = incl = 0
for tx, gd in [(text1, gold1), (text2, gold2)]:
    for did, v in tx.items():
        t = as_text(v)
        for s, e, lab, et in doc_entities(gd, did):
            if et is None: continue
            if t[s:e] == et: excl += 1
            elif t[s:e + 1] == et: incl += 1
print(f"Offset test -> text[s:e]==entity: {excl} | text[s:e+1]==entity: {incl}")
if excl == 0 and incl == 0:
    print("⚠️ Could not verify the convention (no entity-text field matched). Assuming END-EXCLUSIVE.")
END_EXCL = excl >= incl
print("Gold end offset is", "EXCLUSIVE" if END_EXCL else "INCLUSIVE")

# ============================================================
# 2. BIO label space + token labelling / span decoding
# ============================================================
L2I = {"O": 0}
for lab in LABELS:
    L2I["B-" + lab] = len(L2I); L2I["I-" + lab] = len(L2I)
I2L = {v: k for k, v in L2I.items()}
NL = len(L2I)
print(f"Classes: {NL} (O + B/I for each of {len(LABELS)} labels)")

def label_tokens(offsets, ents):
    """ents: [(start, end_exclusive, label)] -> BIO ids per token. Overlapping entities: first (longest) wins."""
    n = len(offsets)
    starts = np.array([o[0] for o in offsets]); ends = np.array([o[1] for o in offsets])
    y = [0] * n; conflicts = 0
    for s, e, lab in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        i = int(np.searchsorted(ends, s, side="right"))   # first token ending after s
        first = True
        while i < n and starts[i] < e:
            if ends[i] > starts[i]:
                if y[i] != 0: conflicts += 1
                else:
                    y[i] = L2I[("B-" if first else "I-") + lab]; first = False
            i += 1
    return y, conflicts

def decode_spans(text, offsets, tags):
    """Token tag ids -> [(start, end_exclusive, label)], whitespace trimmed."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        a, b = offsets[i]
        if b <= a: continue
        name = I2L[int(t)]
        if name == "O":
            if cur: spans.append(cur); cur = None
            continue
        pre, lab = name.split("-", 1)
        if pre == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out

def spans_out(text, offsets, tags):
    return [[int(s), int(e if END_EXCL else e - 1), lab, text[s:e]] for s, e, lab in decode_spans(text, offsets, tags)]

# ============================================================
# 3. Build docs, split by DOCUMENT, windows
# ============================================================
def build_docs(text_d, gold_d, prefix=""):
    docs = []
    for did, v in text_d.items():
        t = as_text(v); enc = tokenizer.encode(t)
        d = {"id": prefix + str(did), "text": t, "ids": enc.ids, "offsets": enc.offsets}
        if gold_d is not None:
            raw = doc_entities(gold_d, did)
            bad = {lab for _, _, lab, _ in raw if lab not in LABELS}
            assert not bad, f"Gold labels not in labels.txt: {bad}"
            d["gold_raw"] = [[s, e, lab] for s, e, lab, _ in raw]
            d["labels"], d["conflicts"] = label_tokens(
                d["offsets"], [(s, e if END_EXCL else e + 1, lab) for s, e, lab, _ in raw])
        docs.append(d)
    return docs

t0 = time.time()
all_docs = build_docs(text1, gold1, "f1:") + build_docs(text2, gold2, "f2:")
test_docs = build_docs(text3, None)
print(f"Tokenized {len(all_docs)} train docs + {len(test_docs)} test docs in {time.time()-t0:.0f}s")

order = list(range(len(all_docs))); random.Random(SEED).shuffle(order)
n_val = max(1, int(VAL_FRACTION * len(all_docs)))
val_docs = [all_docs[i] for i in order[:n_val]]
train_docs = [all_docs[i] for i in order[n_val:]]

def make_windows(n, W=WINDOW, S=STRIDE):
    if n <= W: return [0]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n: starts.append(n - W)
    return starts

train_windows = [(d["ids"][s:s + WINDOW], d["labels"][s:s + WINDOW])
                 for d in train_docs for s in make_windows(len(d["ids"]))]

tok_tr = sum(len(d["ids"]) for d in train_docs)
print(f"Train docs: {len(train_docs)} ({tok_tr:,} tokens) -> {len(train_windows)} windows of ≤{WINDOW} "
      f"(~{sum(len(x) for x,_ in train_windows)/1e6:.2f}M tokens/epoch)")
print(f"Val docs: {len(val_docs)} | Test docs: {len(test_docs)} | "
      f"docs longer than WINDOW: {sum(len(d['ids'])>WINDOW for d in all_docs)} | max tokens: {max(len(d['ids']) for d in all_docs)}")
print(f"Entity-token conflicts (overlapping gold): {sum(d['conflicts'] for d in all_docs)}")
val_cnt = collections.Counter(l for d in val_docs for _, _, l in d["gold_raw"])
print("Val entity counts:", dict(val_cnt.most_common()))

# class weights (sqrt inverse frequency, clipped)
cnt = np.bincount(np.concatenate([np.array(d["labels"]) for d in train_docs]), minlength=NL).astype(np.float64)
w = np.ones(NL)
if USE_CLASS_WEIGHTS:
    for c in range(1, NL): w[c] = float(np.clip(math.sqrt(cnt[0] / max(cnt[c], 1.0)), 1.0, CLASS_W_CLIP))
CLASS_W = torch.tensor(w, dtype=torch.float32, device=device) if USE_CLASS_WEIGHTS else None
print(f"O tokens: {100*cnt[0]/cnt.sum():.1f}% | class weights (min/max entity): {w[1:].min():.1f}/{w[1:].max():.1f}")

def collate_windows(items):
    L = max(len(x) for x, _ in items)
    X = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    Y = torch.full((len(items), L), -100, dtype=torch.long)
    for i, (x, y) in enumerate(items):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X, Y

# ============================================================
# 4. ORACLE round-trip: gold -> token labels -> spans -> strict score  (the ceiling)
# ============================================================
o_docs = train_docs + val_docs
o_gold = {d["id"]: d["gold_raw"] for d in o_docs}
o_pred = {d["id"]: spans_out(d["text"], d["offsets"], d["labels"]) for d in o_docs}
o_res = strict_scores(o_gold, o_pred, LABELS)
print("\n=== ORACLE (ceiling for any model with this tokenizer + labelling) ===")
print_scores(o_res)
_lost = []
for d in o_docs:
    got = {(s, e, l) for s, e, l, _ in o_pred[d["id"]]}
    for s, e, l in d["gold_raw"]:
        if (s, e, l) not in got and len(_lost) < 5: _lost.append((d["id"], s, e, l, d["text"][s:e if END_EXCL else e + 1]))
print("Examples of gold spans the round-trip cannot reproduce:", _lost)
assert o_res["macro_f1"] > 0.8, "Oracle ceiling too low — offsets/tokenizer alignment is off. Paste this output to me."

# ============================================================
# 5. NER wrapper around the GQA decoder (no change to the LM class)
# ============================================================
class NERModel(nn.Module):
    """Token classifier: GQA decoder backbone (causal) + linear head.
    shift=k: token i is classified from the hidden state at position i+k (k-token lookahead)."""
    def __init__(self, lm, d_model, n_labels, dropout=0.1, shift=0):
        super().__init__()
        self.lm = lm
        self.shift = shift
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids):
        lm = self.lm
        x = lm.token_embedding(input_ids)
        for layer in lm.layers:
            x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        out = self.head(self.drop(lm.norm(x)))
        k = self.shift
        if k > 0:   # position i now holds the prediction for token i (made from position i+k)
            out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

# ============================================================
# 6. Prediction (windows averaged) + test-file writer
# ============================================================
@torch.no_grad()
def predict_docs(ner, docs, batch=8):
    ner.eval(); out = {}
    for d in docs:
        n = len(d["ids"])
        probs = torch.zeros(n, NL, device=device)
        starts = make_windows(n)
        for i in range(0, len(starts), batch):
            chunk = starts[i:i + batch]
            seqs = [d["ids"][s:s + WINDOW] for s in chunk]
            L = max(len(q) for q in seqs)
            x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
            for j, q in enumerate(seqs): x[j, :len(q)] = torch.tensor(q)
            with amp(): lg = ner(x.to(device))
            p = F.softmax(lg.float(), dim=-1)
            k = ner.shift
            for j, (s, q) in enumerate(zip(chunk, seqs)):
                valid = len(q) if (s + len(q) >= n) else len(q) - k   # last k tokens of a window have no lookahead
                probs[s:s + valid] += p[j, :valid]
        out[d["id"]] = spans_out(d["text"], d["offsets"], probs.argmax(-1).tolist())
    return out

def write_test_predictions(ner, name):
    preds = predict_docs(ner, test_docs)
    path = f"{P2_DIR}/prediction_{name}.json"
    with open(path, "w", encoding="utf-8") as f: json.dump(preds, f, ensure_ascii=False)
    print(f"Saved {len(preds)} test docs / {sum(len(v) for v in preds.values())} entities -> {path}")
    return path

# ============================================================
# 7. Trainer (optionally with distillation from a teacher NER model)
# ============================================================
def train_ner(ner, name, teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
              backbone_lr=1e-4, head_lr=1e-3, batch_size=8, weight_decay=0.01,
              warmup_frac=0.06, debug=False):
    tw = train_windows[:batch_size * 10] if debug else train_windows
    vdocs = val_docs[:10] if debug else val_docs
    if debug: epochs = 1; name = name + "_debug"
    vgold = {d["id"]: d["gold_raw"] for d in vdocs}
    ner.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)

    groups = {}
    for n_, p in ner.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
         for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / batch_size); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(optimizer, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))

    n_train = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    print(f"[{name}] trainable params: {n_train/1e6:.2f}M | windows: {len(tw)} | steps/epoch: {spe} | "
          f"KD: {'alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None and alpha > 0 else 'off'}")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    best_f1, best_res, bad, history = -1.0, None, 0, []
    best_path = f"{P2_DIR}/{name}_best.pt"; t0 = time.time()

    if teacher is not None: assert teacher.shift == ner.shift, "teacher and student must use the same shift"
    for ep in range(epochs):
        ner.train(); idx = list(range(len(tw))); random.shuffle(idx)
        loss_sum = torch.tensor(0.0, device=device); nb = 0
        bar = tqdm(range(0, len(idx), batch_size), desc=f"{name} ep {ep+1}/{epochs}")
        for i in bar:
            X, Y = collate_windows([tw[j] for j in idx[i:i + batch_size]])
            X, Y = X.to(device), Y.to(device)
            if ner.shift > 0: Y[:, -ner.shift:] = -100        # the last k columns have no lookahead
            with amp(): lg = ner(X)
            lg = lg.float()
            loss = F.cross_entropy(lg.reshape(-1, NL), Y.reshape(-1), weight=CLASS_W, ignore_index=-100)
            if teacher is not None and alpha > 0:
                with torch.no_grad(), amp(): tl = teacher(X)
                m = Y != -100
                kd = F.kl_div(F.log_softmax(lg[m] / T, -1), F.softmax(tl.float()[m] / T, -1),
                              reduction="batchmean") * (T * T)
                loss = (1 - alpha) * loss + alpha * kd
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ner.parameters(), 1.0)
            optimizer.step(); sched.step()
            loss_sum += loss.detach(); nb += 1
            if nb % 20 == 0: bar.set_postfix(loss=f"{(loss_sum / nb).item():.4f}")

        res = strict_scores(vgold, predict_docs(ner, vdocs), LABELS)
        history.append({"epoch": ep + 1, "train_loss": (loss_sum / max(1, nb)).item(), "macro_f1": res["macro_f1"]})
        print(f"epoch {ep+1}: train loss {history[-1]['train_loss']:.4f} | val macro P/R/F1 = "
              f"{res['macro_p']:.4f}/{res['macro_r']:.4f}/{res['macro_f1']:.4f} | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best_f1:
            best_f1, best_res, bad = res["macro_f1"], res, 0
            torch.save({"model": ner.state_dict(), "epoch": ep + 1, "val_macro_f1": best_f1}, best_path)
        else:
            bad += 1
            if bad >= patience: print(f"Early stop (no improvement for {patience} epochs)."); break

    ner.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)["model"])
    mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print(f"✅ [{name}] best val macro-F1 {best_f1:.4f} | {(time.time()-t0)/60:.1f} min | peak mem {mem:.1f} GB | saved {best_path}")
    return {"res": best_res, "history": history, "best_path": best_path, "minutes": (time.time() - t0) / 60,
            "peak_gb": mem, "trainable_params": n_train}

print("\n✅ Phase-2 toolkit ready.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Labels (12): ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Offset test -> text[s:e]==entity: 8393 | text[s:e+1]==entity: 0
Gold end offset is EXCLUSIVE
Classes: 25 (O + B/I for each of 12 labels)
Tokenized 70 train docs + 35 test docs in 2s
Train docs: 60 (399,913 tokens) -> 530 windows of ≤1024 (~0.54M tokens/epoch)
Val docs: 10 | Test docs: 35 | docs longer than WINDOW: 70 | max tokens: 23710
Entity-token conflicts (overlapping gold): 0
Val entity counts: {'PREC': 212, 'STAT': 171, 'DATE': 157, 'AUTH': 150, 'COURT': 97, 'RESP': 89, 'CASENO': 75, 'APP': 66, 'JUDGE': 32, 'WIT': 23, 'R.COUNSEL': 22, 'A.COUNSEL': 20}
O tokens: 90.8% | class weights (min/max entity): 6.1/10.0

=== ORACLE (ceiling for any model with this tokenizer + labelling) ===
MACRO  P=0.9866  R=0.9866  F1=0.9866  | micro F1=0.9819
label            P       R      F1   gold   pred     tp
DATE         1.000   1.000   1.000   1575   1575   1575
STAT         0.

In [ ]:
import glob
for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "alternating_patterns",
             "val_dataset", "train_docs", "NL", "P2_DIR", "print_scores", "strict_scores"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

DEBUG_MODE = False   # True = smoke test (few seconds per block). Set False for the real runs.

SECOND_CKPT =  "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt"      # best checkpoint of the strided model
if not os.path.exists(SECOND_CKPT):
    print("Checkpoint not found. .pt files under the Phase-1 folder:")
    for p in sorted(glob.glob("/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/**/*.pt", recursive=True)):
        print("  ", p, f"{os.path.getsize(p)/1e6:.0f} MB")
    raise FileNotFoundError("Set SECOND_CKPT to the best checkpoint of your strided model (one of the paths above).")

SECOND_PATTERNS = alternating_patterns(10, [(256, 32), (256, 128)])

def build_second_ner(dropout=0.1):
    """Fresh NER model from the PRETRAINED strided checkpoint (patterns must be passed)."""
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=SECOND_PATTERNS)
    ck_ = torch.load(SECOND_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck_["model"] if "model" in ck_ else ck_)
    return NERModel(lm_, d_model=768, n_labels=NL, dropout=dropout).to(device)

# sanity check: Phase-1 val loss with the strided patterns (compare with what you remember for this model)
_m = build_second_ner(); _m.lm.eval(); ls = []
with torch.no_grad():
    for i in range(0, 32, 4):
        ids = torch.stack([val_dataset[j]["input_ids"] for j in range(i, i + 4)]).to(device).long()
        with amp(): ls.append(_m.lm(ids, labels=ids)["loss"].float().item())
print("Layer patterns:", _m.lm.describe_patterns()[:2], "...")
print(f"Second model: Phase-1 val loss ≈ {np.mean(ls):.4f}")
del _m; torch.cuda.empty_cache()

Layer patterns: ['window=256, stride=32', 'window=256, stride=128'] ...
Second model: Phase-1 val loss ≈ 2.3546


In [ ]:
random.seed(1); torch.manual_seed(1)
ner = build_second_ner()
out_full = train_ner(ner, "ner_second_full_ft", epochs=10, patience=3, backbone_lr=1e-4, head_lr=1e-3,
                     batch_size=8, debug=DEBUG_MODE)
print("\n=== Validation (strict) — best epoch ===")
print_scores(out_full["res"])
if not DEBUG_MODE: write_test_predictions(ner, "second_full_ft")
del ner; torch.cuda.empty_cache()

[ner_second_full_ft] trainable params: 87.53M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_full_ft ep 1/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 1.1092 | val macro P/R/F1 = 0.1735/0.3374/0.2221 | 0.1 min


ner_second_full_ft ep 2/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.1466 | val macro P/R/F1 = 0.1930/0.3430/0.2424 | 0.3 min


ner_second_full_ft ep 3/10:   0%|          | 0/67 [00:00<?, ?it/s]

... (11 outputs left out)

ner_second_full_ft ep 9/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 9: train loss 0.0024 | val macro P/R/F1 = 0.2957/0.3529/0.3111 | 1.3 min


ner_second_full_ft ep 10/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 10: train loss 0.0018 | val macro P/R/F1 = 0.2938/0.3607/0.3137 | 1.5 min
Early stop (no improvement for 3 epochs).
✅ [ner_second_full_ft] best val macro-F1 0.3201 | 1.5 min | peak mem 13.0 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_full_ft_best.pt

=== Validation (strict) — best epoch ===
MACRO  P=0.3021  R=0.3596  F1=0.3201  | micro F1=0.3920
label            P       R      F1   gold   pred     tp
PREC         0.171   0.241   0.200    212    298     51
STAT         0.480   0.567   0.520    171    202     97
DATE         0.914   0.949   0.931    157    163    149
AUTH         0.178   0.107   0.133    150     90     16
COURT        0.574   0.722   0.639     97    122     70
RESP         0.140   0.067   0.091     89     43      6
CASENO       0.260   0.427   0.323     75    123     32
APP          0.157   0.121   0.137     66     51      8
JUDGE        0.455   0.469   0.462     32     33     15
WIT          0.000   0.000   0.000     23     16      

In [ ]:
class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B (A x)"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

def inject_adapters(ner, cls, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in ner.parameters(): p.requires_grad_(False)
    for layer in ner.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    for p in ner.head.parameters(): p.requires_grad_(True)

@torch.no_grad()
def merge_diff(mod):
    W0 = mod.base.weight; x = torch.randn(4, W0.shape[1], device=W0.device)
    Wm = W0 + mod.scale * (mod.B @ mod.A) if isinstance(mod, LoRALinear) else W0 + (mod.scale * (mod.B @ mod.A)) @ W0
    return (mod(x) - F.linear(x, Wm)).abs().max().item()

LORA_RESULTS = []
for lr in [1e-3, 3e-3]:
    random.seed(1); torch.manual_seed(1)
    ner = build_second_ner()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    ner.eval()
    with torch.no_grad(): before = ner(ids)
    inject_adapters(ner, LoRALinear, r=8, alpha=16)
    with torch.no_grad(): after = ner(ids)
    d = (before - after).abs().max().item(); assert d < 1e-4, f"not identity at init ({d})"
    print(f"[lora] identity-at-init OK ({d:.1e})")
    out = train_ner(ner, f"ner_second_lora_lr{lr:g}", epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, debug=DEBUG_MODE)
    print(f"[lora] merge check: {merge_diff(ner.lm.layers[0].attn.q_proj):.1e}")
    if not DEBUG_MODE: write_test_predictions(ner, f"second_lora_lr{lr:g}")
    LORA_RESULTS.append((lr, out["trainable_params"], max(h["macro_f1"] for h in out["history"]),
                         out["res"]["macro_p"], out["res"]["macro_r"], out["minutes"], out["peak_gb"]))
    del ner; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in LORA_RESULTS: print(f"{'lora':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")

[lora] identity-at-init OK (0.0e+00)
[ner_second_lora_lr0.001] trainable params: 0.43M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_lora_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.3447 | val macro P/R/F1 = 0.0595/0.2062/0.0901 | 0.1 min


ner_second_lora_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.3775 | val macro P/R/F1 = 0.1384/0.2996/0.1847 | 0.3 min


ner_second_lora_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (103 outputs left out)

ner_second_lora_lr0.003 ep 29/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 29: train loss 0.0009 | val macro P/R/F1 = 0.2580/0.3370/0.2868 | 4.1 min


ner_second_lora_lr0.003 ep 30/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 30: train loss 0.0008 | val macro P/R/F1 = 0.2694/0.3347/0.2929 | 4.3 min
✅ [ner_second_lora_lr0.003] best val macro-F1 0.2929 | 4.3 min | peak mem 11.2 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_lora_lr0.003_best.pt
[lora] merge check: 2.9e-06
Saved 35 test docs / 5145 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_second_lora_lr0.003.json

method       lr  trainable  best F1       P       R   min    GB
lora      1e-03    428,825   0.2993  0.2723  0.3460   3.7  11.2
lora      3e-03    428,825   0.2929  0.2694  0.3347   4.3  11.2


In [ ]:
class LoRMAPlusLinear(nn.Module):
    """LoRMA+: W = ((alpha/r) B A + I) W0, B,A on the OUTPUT side. y = W0 x; out = y + (alpha/r) B (A y)"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_adapters(ner, cls, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in ner.parameters(): p.requires_grad_(False)
    for layer in ner.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    for p in ner.head.parameters(): p.requires_grad_(True)

@torch.no_grad()
def merge_diff(mod):
    W0 = mod.base.weight; x = torch.randn(4, W0.shape[1], device=W0.device)
    Wm = W0 + (mod.scale * (mod.B @ mod.A)) @ W0          # (I + s·BA) W0
    return (mod(x) - F.linear(x, Wm)).abs().max().item()

LORMA_RESULTS = []
for lr in [1e-3, 3e-3]:
    random.seed(1); torch.manual_seed(1)
    ner = build_second_ner()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    ner.eval()
    with torch.no_grad(): before = ner(ids)
    inject_adapters(ner, LoRMAPlusLinear, r=10, alpha=20)
    with torch.no_grad(): after = ner(ids)
    d = (before - after).abs().max().item(); assert d < 1e-4, f"not identity at init ({d})"
    print(f"[lorma] identity-at-init OK ({d:.1e})")
    out = train_ner(ner, f"ner_second_lorma_lr{lr:g}", epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, debug=DEBUG_MODE)
    print(f"[lorma] merge check: {merge_diff(ner.lm.layers[0].attn.q_proj):.1e}")
    if not DEBUG_MODE: write_test_predictions(ner, f"second_lorma_lr{lr:g}")
    LORMA_RESULTS.append((lr, out["trainable_params"], max(h["macro_f1"] for h in out["history"]),
                          out["res"]["macro_p"], out["res"]["macro_r"], out["minutes"], out["peak_gb"]))
    del ner; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in LORMA_RESULTS: print(f"{'lorma':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")

[lorma] identity-at-init OK (0.0e+00)
[ner_second_lorma_lr0.001] trainable params: 0.43M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_lorma_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.3048 | val macro P/R/F1 = 0.0551/0.1939/0.0841 | 0.1 min


ner_second_lorma_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.3717 | val macro P/R/F1 = 0.1509/0.3099/0.1986 | 0.3 min


ner_second_lorma_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (57 outputs left out)

ner_second_lorma_lr0.003 ep 13/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 13: train loss 0.0467 | val macro P/R/F1 = 0.1916/0.3286/0.2359 | 1.8 min


ner_second_lorma_lr0.003 ep 14/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 14: train loss 0.0384 | val macro P/R/F1 = 0.2080/0.3319/0.2476 | 1.9 min
Early stop (no improvement for 6 epochs).
✅ [ner_second_lorma_lr0.003] best val macro-F1 0.2679 | 2.0 min | peak mem 11.1 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_lorma_lr0.003_best.pt
[lorma] merge check: 1.1e-06
Saved 35 test docs / 6852 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_second_lorma_lr0.003.json

method       lr  trainable  best F1       P       R   min    GB
lorma     1e-03    428,825   0.2806  0.2489  0.3361   2.6  11.1
lorma     3e-03    428,825   0.2679  0.2162  0.3785   2.0  11.1


In [ ]:
PREFIX_LEN = 32

class PrefixNERModel(NERModel):
    """Frozen backbone + trainable per-layer K/V prefixes (+ NER head), fed through past_kv."""
    def __init__(self, lm, d_model, n_labels, init_ids, dropout=0.1, shift=0):
        super().__init__(lm, d_model, n_labels, dropout, shift)
        for p in self.parameters(): p.requires_grad_(False)
        lm.eval()
        with torch.no_grad():
            dev = next(lm.parameters()).device
            pkv = lm(torch.tensor([init_ids], device=dev))["past_key_values"]
        self.prefix_k = nn.ParameterList([nn.Parameter(k.detach().clone()) for k, _ in pkv])
        self.prefix_v = nn.ParameterList([nn.Parameter(v.detach().clone()) for _, v in pkv])
        for p in self.head.parameters(): p.requires_grad_(True)
    def hidden(self, input_ids):
        lm = self.lm; B = input_ids.shape[0]
        x = lm.token_embedding(input_ids)
        for i, layer in enumerate(lm.layers):
            pkv = (self.prefix_k[i].expand(B, -1, -1, -1), self.prefix_v[i].expand(B, -1, -1, -1))
            x, _ = layer(x, lm.cos, lm.sin, past_kv=pkv)
        return lm.norm(x)
    def forward(self, input_ids):
        out = self.head(self.drop(self.hidden(input_ids)))
        k = self.shift
        if k > 0: out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

@torch.no_grad()
def plain_hidden(lm, ids):
    x = lm.token_embedding(ids)
    for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
    return lm.norm(x)

PREFIX_RESULTS = []
for lr in [1e-3, 5e-3]:
    random.seed(1); torch.manual_seed(1)
    base = build_second_ner()
    init_ids = train_docs[0]["ids"][:PREFIX_LEN]
    ner = PrefixNERModel(base.lm, d_model=768, n_labels=NL, init_ids=init_ids).to(device)
    ner.eval()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    with torch.no_grad():
        h_prefix = ner.hidden(ids)
        h_full = plain_hidden(ner.lm, torch.cat([torch.tensor([init_ids], device=device), ids], dim=1))[:, PREFIX_LEN:]
    d = (h_prefix - h_full).abs().max().item(); assert d < 1e-3, f"prefix path mismatch ({d})"
    print(f"[prefix] init check OK ({d:.1e})")
    out = train_ner(ner, f"ner_second_prefix_lr{lr:g}", epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, weight_decay=0.0, debug=DEBUG_MODE)
    if not DEBUG_MODE: write_test_predictions(ner, f"second_prefix_lr{lr:g}")
    PREFIX_RESULTS.append((lr, out["trainable_params"], max(h["macro_f1"] for h in out["history"]),
                           out["res"]["macro_p"], out["res"]["macro_r"], out["minutes"], out["peak_gb"]))
    del ner, base; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in PREFIX_RESULTS: print(f"{'prefix':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")

[prefix] init check OK (2.0e-05)
[ner_second_prefix_lr0.001] trainable params: 0.18M | windows: 530 | steps/epoch: 67 | KD: off


ner_second_prefix_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.6633 | val macro P/R/F1 = 0.0096/0.1004/0.0175 | 0.1 min


ner_second_prefix_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.7808 | val macro P/R/F1 = 0.0524/0.2363/0.0844 | 0.3 min


ner_second_prefix_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (111 outputs left out)

ner_second_prefix_lr0.005 ep 29/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 29: train loss 0.2132 | val macro P/R/F1 = 0.1252/0.3012/0.1728 | 4.0 min


ner_second_prefix_lr0.005 ep 30/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 30: train loss 0.2113 | val macro P/R/F1 = 0.1246/0.3012/0.1724 | 4.1 min
✅ [ner_second_prefix_lr0.005] best val macro-F1 0.1751 | 4.1 min | peak mem 11.0 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_second_prefix_lr0.005_best.pt
Saved 35 test docs / 11651 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_second_prefix_lr0.005.json

method       lr  trainable  best F1       P       R   min    GB
prefix    1e-03    183,065   0.1496  0.1044  0.2933   4.1  11.0
prefix    5e-03    183,065   0.1751  0.1269  0.3048   4.1  11.0
